# Week 2: Sampling Configuration Experiment

This notebook supports the Week 2 discussion assignment by running the **same prompt through the same Gemini model three times** while changing the requested sampling configuration.

The three configurations are:

1. Temperature `0`
2. Temperature `0.7` with top-p `0.9`
3. Temperature `1.2` with top-k `50`

The generated outputs are preserved exactly as produced for comparison.


## Prompt and task

**Prompt**

> Explain how a large language model generates the next token in a response. Write the explanation for a non-technical audience in no more than 100 words.

**Task**

The prompt asks the model to explain next-token generation accurately, concisely, and in language understandable to a non-technical audience. This makes it possible to compare factual accuracy, style, length, and clarity as sampling changes.


## Setup

This notebook uses `gemini-3.1-flash-lite` through Google's `google-genai` Python SDK. The Gemini API exposes temperature, top-p, and top-k generation controls directly.

Store the API key as `GEMINI_API_KEY`. In Google Colab, add it through **Secrets** rather than placing the key in the notebook. The fallback environment-variable path also works when running locally.


In [13]:
# In Google Colab, uncomment this line on the first run:
# !pip install -q google-genai

import os
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.1-flash-lite"

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, KeyError):
    api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY was not found. Add it to Colab Secrets or your local environment."
    )

client = genai.Client(api_key=api_key)
print("Model:", MODEL_NAME)


Model: gemini-3.1-flash-lite


## Prepare the identical prompt

The prompt is defined once and reused for all three runs so the sampling configuration is the experimental variable.


In [14]:
PROMPT = (
    "Explain how a large language model generates the next token in a response. "
    "Write the explanation for a non-technical audience in no more than 100 words."
)

print(PROMPT)


Explain how a large language model generates the next token in a response. Write the explanation for a non-technical audience in no more than 100 words.


## Generation helper

The helper sends the same prompt to Gemini with the requested generation configuration. Each experiment cell prints the response exactly as returned and stores it for later comparison.


In [15]:
def generate_response(**sampling_settings):
    config = types.GenerateContentConfig(
        max_output_tokens=160,
        **sampling_settings,
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=PROMPT,
        config=config,
    )

    text = response.text
    print(text)
    return text


## Run 1: Temperature 0

This run sets temperature to zero, strongly favoring the model's highest-probability token choices. Other sampling parameters are left at Gemini's defaults.


In [16]:
output_temperature_0 = generate_response(
    temperature=0.0,
)


Think of an LLM as a super-powered autocomplete. It doesn’t "know" facts; it calculates probabilities. 

When you type a prompt, the model converts your words into numbers. It analyzes these numbers against billions of patterns learned during training to predict which word (or "token") is most likely to come next. It picks one, adds it to your original text, and repeats the process. By constantly choosing the most statistically probable next piece, it builds coherent sentences. It’s essentially playing a massive game of "guess the next word" based on everything it has ever read.


## Run 2: Temperature 0.7 with top-p 0.9

This run raises temperature and sets nucleus sampling to a cumulative probability threshold of 0.9. Top-k is left at the model default because it is not part of this requested configuration.


In [17]:
output_temperature_07_top_p_09 = generate_response(
    temperature=0.7,
    top_p=0.9,
)


Think of an LLM as a super-advanced version of your phone’s autocomplete. It doesn’t "think" like a human; instead, it uses math to predict patterns. 

When you type a prompt, the model converts your words into numerical data. It analyzes billions of sentences it studied during training to calculate the probability of which word (or "token") should come next. It selects one, adds it to your original text, and repeats the process. By constantly choosing the most statistically likely next piece, it builds a coherent, human-like response one small step at a time.


## Run 3: Temperature 1.2 with top-k 50

This run increases randomness and limits consideration to the 50 highest-probability tokens through top-k. Top-p is left at the model default because it is not part of this requested configuration.


In [19]:
output_temperature_12_top_k_50 = generate_response(
    temperature=1.2,
    top_k=50,
)


Imagine a large language model as a super-advanced version of your phone’s autocomplete. It doesn't "think" like a human; instead, it has studied billions of sentences to learn patterns in language.

When you ask a question, the model looks at your words and calculates the statistical probability of every word that could come next. It assigns a "likelihood score" to these options based on its training. It then randomly selects one of the most probable candidates to be the next token. It repeats this process word-by-word until your entire response is complete.


## Preserve and compare the outputs

The three responses above are the exact outputs used for this analysis. Sampling can produce different text on subsequent calls, so rerunning a stochastic cell could produce different wording.


In [20]:
results = {
    "Temperature 0": output_temperature_0,
    "Temperature 0.7, top-p 0.9": output_temperature_07_top_p_09,
    "Temperature 1.2, top-k 50": output_temperature_12_top_k_50,
}

for configuration, response in results.items():
    word_count = len(response.split())
    print(f"{configuration}: {word_count} words")


Temperature 0: 94 words
Temperature 0.7, top-p 0.9: 93 words
Temperature 1.2, top-k 50: 92 words


## Analysis notes

### 1. What prompt did you use, and what task was it meant to accomplish?

The prompt was: **“Explain how a large language model generates the next token in a response. Write the explanation for a non-technical audience in no more than 100 words.”** The task was to produce a short, understandable explanation of next-token generation while giving enough room for different sampling settings to change the wording.

### 2. How did the outputs differ in style, length, and accuracy?

All three responses used an autocomplete analogy and remained broadly accurate, but their phrasing changed. Temperature 0 was the most deterministic and direct, describing the process as a “massive game of guess the next word.” Temperature 0.7 with top-p 0.9 sounded smoother and more conversational, using the familiar phone-autocomplete comparison. Temperature 1.2 with top-k 50 was slightly more explicit about candidate probabilities and random selection, but it also simplified token generation as happening “word-by-word.”

The lengths were very close: **94 words** at temperature 0, **93 words** at temperature 0.7/top-p 0.9, and **92 words** at temperature 1.2/top-k 50. All three satisfied the 100-word constraint. None contained a major factual error, although each simplified the process for a non-technical audience.

### 3. Which configuration was best for your task, and why?

**Temperature 0.7 with top-p 0.9** was the best fit. It preserved the core explanation, stayed within the length limit, and sounded the most natural for a non-technical audience without becoming noticeably less accurate.

### 4. Which stage of the inference pipeline is most responsible for the differences you observed?

The differences primarily came from **next-token sampling/selection**. After the model produces scores for possible next tokens, temperature changes how strongly higher-probability choices are favored, top-p limits candidates according to cumulative probability, and top-k limits the candidate set to a fixed number of the most probable tokens. Those controls changed which tokens were selected and therefore changed the wording and style of each response.

### 5. If you had to explain the worst output to a non-technical stakeholder, how would you describe what went wrong?

The weakest response was the **temperature 1.2 with top-k 50** output, although it was still usable. I would explain that the more aggressive sampling setting gave the model more freedom in how it worded the explanation. That produced a slightly looser description, especially the phrase “word-by-word,” because models actually generate tokens, which are not always whole words. The problem was not that the model lost the topic; the wording simply became a little less precise as randomness increased.
